# Textvektorisierung

In diesem Notebook werden die vorverarbeiteten Beschwerdetexte in numerische Vektoren umgewandelt. Dafür werden die Verfahren Bag-of-Words (BoW) und TF-IDF verwendet und anschließend miteinander verglichen.

## Import der benötigten Bibliotheken

Für die Vektorisierung der Texte werden Verfahren aus der Bibliothek scikit-learn verwendet.

In [35]:
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

## Laden der vorverarbeiteten Texte

Die in Notebook 2 erzeugten bereinigten und lemmatisierten Texte werden geladen und für die Vektorisierung verwendet.

In [36]:
df = pd.read_csv("../processed_data/02_preprocessed_data.csv")

In [37]:
df[['processed_text']].head()

,processed_text
0,account
1,kindly address issue credit report assert acco...
2,write three request unverified account list st...
3,xxxx xxxx old account settle xxxx keep reappea...
4,call hour weekend use various number


## Vektorisierung mit Bag-of-Words (BoW)

Beim Bag-of-Words-Verfahren wird gezählt, wie häufig Wörter innerhalb eines Dokuments vorkommen. Die Reihenfolge der Wörter wird dabei nicht berücksichtigt.

In [38]:
bow_vectorizer = CountVectorizer(max_features=1000)

In [39]:
X_bow = bow_vectorizer.fit_transform(df['processed_text'])

## Analyse der Bag-of-Words-Vektoren

Die wichtigsten Begriffe des Bag-of-Words-Modells werden angezeigt.

In [40]:
bow_words = bow_vectorizer.get_feature_names_out()

print(bow_words[:20])

['ability' 'able' 'absolutely' 'abuse' 'accept' 'access' 'accord'
 'accordance' 'accordingly' 'account' 'accounting' 'acct' 'accuracy'
 'accurate' 'accurately' 'acknowledge' 'across' 'act' 'action' 'active']


In [41]:
X_bow.shape

(5000, 1000)

## Vektorisierung mit TF-IDF

TF-IDF bewertet Wörter nicht nur nach ihrer Häufigkeit, sondern zusätzlich nach ihrer Relevanz innerhalb des gesamten Textkorpus.

In [42]:
tfidf_vectorizer = TfidfVectorizer(max_features=1000)

In [43]:
X_tfidf = tfidf_vectorizer.fit_transform(df['processed_text'])

## Analyse der TF-IDF-Vektoren

Die wichtigsten Begriffe des TF-IDF-Modells werden angezeigt.

In [44]:
tfidf_words = tfidf_vectorizer.get_feature_names_out()

print(tfidf_words[:20])

['ability' 'able' 'absolutely' 'abuse' 'accept' 'access' 'accord'
 'accordance' 'accordingly' 'account' 'accounting' 'acct' 'accuracy'
 'accurate' 'accurately' 'acknowledge' 'across' 'act' 'action' 'active']


In [45]:
X_tfidf.shape

(5000, 1000)

## Vergleich von Bag-of-Words und TF-IDF

Bag-of-Words berücksichtigt ausschließlich die Häufigkeit von Wörtern innerhalb eines Dokuments. TF-IDF bewertet zusätzlich die Relevanz eines Wortes im gesamten Datensatz und reduziert dadurch den Einfluss sehr häufiger Begriffe. Dadurch liefert TF-IDF häufig präzisere Ergebnisse für semantische Analysen und Topic Modeling.

Ein konkreter Vergleich der beiden Verfahren anhand der erzeugten Ergebnisse erfolgt im nächsten Schritt.

In [46]:
import numpy as np

bow_sum = X_bow.sum(axis=0)

bow_freq = [
    (word, bow_sum[0, idx])
    for word, idx in bow_vectorizer.vocabulary_.items()
]

bow_freq = sorted(bow_freq, key=lambda x: x[1], reverse=True)

bow_freq[:10]

[('xxxx', np.int64(59301)),
 ('credit', np.int64(10223)),
 ('account', np.int64(9992)),
 ('report', np.int64(9254)),
 ('information', np.int64(5848)),
 ('consumer', np.int64(4953)),
 ('xxxxxxxx', np.int64(4759)),
 ('reporting', np.int64(3578)),
 ('payment', np.int64(3541)),
 ('dispute', np.int64(2699))]

In [47]:
tfidf_sum = X_tfidf.sum(axis=0)

tfidf_freq = [
    (word, tfidf_sum[0, idx])
    for word, idx in tfidf_vectorizer.vocabulary_.items()
]

tfidf_freq = sorted(tfidf_freq, key=lambda x: x[1], reverse=True)

tfidf_freq[:10]

[('xxxx', np.float64(1198.3916808016936)),
 ('report', np.float64(362.10375177161404)),
 ('credit', np.float64(361.6930174907256)),
 ('account', np.float64(361.44325514938936)),
 ('information', np.float64(225.83358306087604)),
 ('consumer', np.float64(202.9353387341531)),
 ('payment', np.float64(177.46421129306222)),
 ('xxxxxxxx', np.float64(168.1775471033859)),
 ('reporting', np.float64(165.56452912636584)),
 ('usc', np.float64(144.68917503614892))]

## Vergleich von Bag-of-Words und TF-IDF

Beide Verfahren erzeugten eine Matrixgröße von `(5000, 1000)`. Dies bedeutet, dass 5000 Dokumente anhand von 1000 relevanten Textmerkmalen analysiert wurden.

Beim Bag-of-Words-Verfahren dominierten besonders häufig vorkommende Begriffe wie `xxxx`, `credit`, `account` und `report`. Da BoW ausschließlich die Worthäufigkeit betrachtet, erhalten sehr häufig vorkommende Wörter automatisch ein hohes Gewicht. Dadurch können allgemeine oder wenig aussagekräftige Begriffe die Analyse stark beeinflussen.

TF-IDF verwendet dagegen zusätzlich eine Gewichtung auf Basis der Relevanz eines Wortes innerhalb des gesamten Datensatzes. Wörter, die in nahezu allen Dokumenten vorkommen, werden schwächer gewichtet, während spezifischere Begriffe stärker hervorgehoben werden. Dies zeigt sich beispielsweise am Begriff `usc`, der im TF-IDF-Modell stärker berücksichtigt wird als im Bag-of-Words-Verfahren.

Insgesamt liefert TF-IDF präzisere semantische Informationen und eignet sich daher besser für die spätere Themenanalyse und das Topic Modeling.

## Speicherung der Vektorisierungsergebnisse

Die erzeugten Textvektoren werden gespeichert, damit sie für die folgenden Schritte der Themenanalyse erneut verwendet werden können. Aufgrund der komplexen Matrixstrukturen werden die Daten als Pickle-Dateien gespeichert.

In [48]:
import pickle

In [49]:
with open("../processed_data/03_bow_vectors.pkl", "wb") as f:
    pickle.dump(X_bow, f)

with open("../processed_data/03_tfidf_vectors.pkl", "wb") as f:
    pickle.dump(X_tfidf, f)

In [50]:
with open("../processed_data/03_bow_vectorizer.pkl", "wb") as f:
    pickle.dump(bow_vectorizer, f)

with open("../processed_data/03_tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)